In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.silver

In [0]:
# 1. Read the bronze customer table
import pyspark.sql.functions as F
df_raw = spark.table("bronze.customers")

# 2. Normalize string values so near-duplicates collapse to exact duplicates
#    (trim, collapse repeated whitespace; case-insensitive for matching)
string_cols = [f.name for f in df_raw.schema.fields if f.dataType.simpleString() == "string"]
df = df_raw
for c in string_cols:
    df = df.withColumn(c, F.regexp_replace(F.trim(F.col(c)), r"\s+", " "))

# dedup key: customer_id ignoring case/whitespace differences
df = df.withColumn("_dedup_key", F.lower(F.col("customer_id")))


# 3. Check for duplicates (exact + near, now normalized)
duplicates = (
    df.groupBy("_dedup_key")
    .agg(F.count("*").alias("count"), F.collect_set("customer_id").alias("raw_variants"))
    .filter("count > 1")
    .orderBy(F.col("count").desc())
)
print(f"Total rows in bronze.customers: {df_raw.count()}")
print(f"Distinct customer_ids (normalized): {df.select('_dedup_key').distinct().count()}")
dup_count = duplicates.count()
print(f"customer_ids with duplicates (incl. near-duplicates): {dup_count}")

if dup_count > 0:
    print("\nDuplicate customer_ids (raw variants shown):")
    duplicates.show(truncate=False)

# 4. Remove duplicates, keeping one row per normalized customer_id
#    (dropDuplicates keeps the first occurrence for each key)
df_deduped = df.dropDuplicates(["_dedup_key"]).drop("_dedup_key")

df_deduped.display()


In [0]:
# 3b. Normalize column names: trim leading/trailing spaces and lowercase
df_deduped = df_deduped.toDF(*[c.strip().lower() for c in df_deduped.columns])

print("Normalized column names:")
print(df_deduped.columns)


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS QUARANTINE

In [0]:
# 3c. Complete missing / invalid values for customer_id, age, gender, loyalty_member

# Cast age to numeric so validity checks work
missing_customer_id = F.col("customer_id").isNull() | (F.trim(F.col("customer_id")) == "")
invalid_age = F.col("age").isNull() | ~F.col("age").cast("double").between(0, 120)
missing_loyalty = F.col("loyalty_member").isNull() | (F.trim(F.col("loyalty_member")) == "")

df_deduped = df_deduped.withColumn("age", F.col("age").cast("double"))
# Re-evaluate age condition now that age is numeric
invalid_age = F.col("age").isNull() | ~F.col("age").between(0, 120)

# Report missing/invalid counts before cleaning
print("Before cleaning:")
df_deduped.select(
    F.sum(F.when(missing_customer_id, 1).otherwise(0)).alias("missing_customer_id"),
    F.sum(F.when(invalid_age, 1).otherwise(0)).alias("invalid_age"),
    F.sum(F.when(F.col("gender").isNull() | (F.trim(F.col("gender")) == ""), 1).otherwise(0)).alias("missing_gender"),
    F.sum(F.when(missing_loyalty, 1).otherwise(0)).alias("missing_loyalty_member"),
).show()

# --- flag quarantine rows: missing customer_id, invalid age (not between 0-120), or missing loyalty_member ---
# (flag computed BEFORE any fills, so it reflects the raw values)
df_deduped = df_deduped.withColumn(
    "_quarantine_reason",
    F.concat_ws(
        ";",
        F.when(missing_customer_id, F.lit("missing_customer_id")),
        F.when(invalid_age, F.lit("invalid_age")),
        F.when(missing_loyalty, F.lit("missing_loyalty_member")),
    ),
)

# --- create/append quarantined rows (raw values) to quarantine.customers ---
quarantine_df = df_deduped.filter(F.col("_quarantine_reason") != "").drop("_quarantine_reason")
quarantine_count = quarantine_df.count()
if quarantine_count > 0:
    quarantine_df.write.mode("append").saveAsTable("quarantine.customers")
    print(f"Appended {quarantine_count} quarantined rows to quarantine.customers")
else:
    print("No rows to quarantine")

# --- gender: normalize to M/F, unrecognized -> 'unknown' (rows kept) ---
df_deduped = df_deduped.withColumn(
    "gender",
    F.when(F.lower(F.col("gender")).isin("m", "male"), "M")
     .when(F.lower(F.col("gender")).isin("f", "female"), "F")
     .otherwise("unknown"),
)

# --- loyalty_member: fill missing/blank with a deterministic value derived from customer_id ---
# (no median/imputation for age; invalid ages are quarantined and removed instead)
df_deduped = df_deduped.withColumn(
    "loyalty_member",
    F.when(missing_loyalty, F.concat(F.lit("LOY-"), F.upper(F.col("customer_id"))))
     .otherwise(F.col("loyalty_member")),
)

# --- final dataframe: drop rows with missing customer_id or invalid age ---
df_deduped = df_deduped.filter(~missing_customer_id & ~invalid_age).drop("_quarantine_reason")

print("After cleaning:")
df_deduped.select(
    F.sum(F.when(missing_customer_id, 1).otherwise(0)).alias("missing_customer_id"),
    F.sum(F.when(invalid_age, 1).otherwise(0)).alias("invalid_age"),
    F.sum(F.when(F.col("gender") == "unknown", 1).otherwise(0)).alias("unknown_gender"),
    F.sum(F.when(F.col("loyalty_member").isNull() | (F.trim(F.col("loyalty_member")) == ""), 1).otherwise(0)).alias("missing_loyalty_member"),
).show()

df_deduped.display()

In [0]:
%sql
drop materialized view workspace.silver.customers


In [0]:
# 5. Write the cleaned dataframe to the silver layer
(df_deduped.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.customers"))

# Verify the write
print(f"Rows written to workspace.silver.customers: {spark.table('workspace.silver.customers').count()}")
spark.table("workspace.silver.customers").display()

In [0]:
%sql
select * from quarantine.customers